# Pandas Data Manipulation Cheatsheet
**Basics → Intermediate → Advanced** — every cell is runnable.

| Part | Topic |
|------|-------|
| A | Fundamentals (creating, inspecting, selecting, missing data, duplicates) |
| B | Core Patterns (filtering, groupby, windows, pivot, stats, merging) |
| C | Advanced (strings, datetime, apply/map, method chaining) |
| D | Interview Decision Tree & Quick Reference |


## Setup — run this first

In [ ]:
import pandas as pd
import numpy as np
np.random.seed(42)

# Shared demo DataFrame used throughout this cheatsheet
df = pd.DataFrame({
    'dept':   ['Eng','Eng','Eng','Sales','Sales','HR','HR'],
    'name':   ['Alice','Bob','Carol','Dave','Eve','Frank','Grace'],
    'salary': [90, 80, 70, 85, 95, 75, 88],
    'score':  [4.5, 3.8, 4.2, 3.9, 4.7, 3.5, 4.1],
    'city':   ['NY','SF','NY','NY','SF','Chicago','SF'],
    'hire_date': pd.to_datetime(['2019-01-15','2020-03-20','2021-06-10',
                                  '2018-02-05','2022-04-18','2020-07-22','2019-11-01']),
    'age':    [32, 28, 35, 45, 26, 38, 31]
})
print("df shape:", df.shape)
df

---
## Part A: Fundamentals

### A1. Creating DataFrames & Series
| Pattern | Syntax |
|---------|--------|
| From dict | `pd.DataFrame({'a': [1,2], 'b': [3,4]})` |
| From list of dicts | `pd.DataFrame([{'a':1,'b':2}, {'a':3,'b':4}])` |
| From CSV | `pd.read_csv('file.csv')` |
| Series | `pd.Series([1,2,3], name='col')` |
| Date range | `pd.date_range('2024-01-01', periods=10)` |

**`read_csv` common params:** `sep`, `header`, `index_col`, `usecols`, `dtype`, `parse_dates`, `na_values`, `nrows`


In [ ]:
# Various creation methods
from_dict       = pd.DataFrame({'a': [1,2,3], 'b': [4,5,6]})
from_list_dicts = pd.DataFrame([{'x': 10, 'y': 20}, {'x': 30, 'y': 40}])
series_example  = pd.Series([1.0, 2.5, 3.7], name='values')
date_range      = pd.date_range('2024-01-01', periods=5, freq='MS')  # month start

print("from_dict:\n", from_dict)
print("\nfrom_list_dicts:\n", from_list_dicts)
print("\ndate_range:", date_range.tolist())

### A2. Inspecting Data
| Pattern | Syntax |
|---------|--------|
| Shape | `df.shape` |
| Data types | `df.dtypes` |
| Info | `df.info()` |
| Head / Tail | `df.head(5)` / `df.tail(5)` |
| Unique values | `df['col'].unique()` |
| Value counts | `df['col'].value_counts()` |
| Describe | `df.describe()` |
| NaN count | `df.isna().sum()` |
| Duplicate count | `df.duplicated().sum()` |

> **Interview tip:** Start every data question with shape, dtypes, and null counts.


In [ ]:
print("Shape:", df.shape)
print("\nDtypes:\n", df.dtypes)
print("\nDept value_counts:\n", df['dept'].value_counts())
print("\nNaN counts:\n", df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())
df.describe()

### A3. Selecting & Indexing
| Pattern | Syntax |
|---------|--------|
| Single column | `df['col']` (Series) |
| Multiple columns | `df[['col1', 'col2']]` (DataFrame) |
| By label | `df.loc[row_label, col_label]` |
| By position | `df.iloc[row_pos, col_pos]` |
| Conditional rows | `df.loc[df['col'] > 5]` |
| Select dtypes | `df.select_dtypes(include='number')` |

**`loc` uses labels, end is INCLUSIVE. `iloc` uses positions, end is EXCLUSIVE.**


In [ ]:
# Single column vs multiple
print("Single column (Series):")
print(df['name'].head(3))

print("\nMultiple columns (DataFrame):")
print(df[['name', 'salary']].head(3))

print("\nloc by label:  ", df.loc[0, 'name'])
print("iloc by pos:   ", df.iloc[0, 1])

print("\nConditional — salary > 85:")
print(df.loc[df['salary'] > 85, ['name', 'dept', 'salary']])

print("\nNumeric columns only:")
print(df.select_dtypes(include='number').head(3))

### A4. Adding, Renaming & Dropping Columns
| Pattern | Syntax |
|---------|--------|
| Add column | `df['new'] = df['a'] + df['b']` |
| Assign (chainable) | `df.assign(new=lambda x: x['a'] + x['b'])` |
| Rename columns | `df.rename(columns={'old': 'new'})` |
| Lowercase all cols | `df.columns = df.columns.str.lower()` |
| Drop column | `df.drop(columns=['col'])` |
| Map/Replace values | `df['col'].map({'a': 1, 'b': 2})` |


In [ ]:
result = (df.copy()
    .assign(bonus = lambda x: x['salary'] * 0.1)
    .assign(total = lambda x: x['salary'] + x['bonus'])
    .rename(columns={'dept': 'department'})
    .drop(columns=['age'])
)
result[['name', 'department', 'salary', 'bonus', 'total']].head(4)

### A5. Sorting & Reindexing
| Pattern | Syntax |
|---------|--------|
| Sort by column | `df.sort_values('col', ascending=False)` |
| Sort by multiple | `df.sort_values(['a','b'], ascending=[True, False])` |
| Reset index | `df.reset_index(drop=True)` |
| nlargest / nsmallest | `df.nlargest(5, 'col')` |

> `nlargest` is faster than `sort_values + head` for large DataFrames.


In [ ]:
print("Top 3 by salary:")
print(df.nlargest(3, 'salary')[['name', 'dept', 'salary']])

print("\nSort by dept asc, salary desc:")
print(df.sort_values(['dept', 'salary'], ascending=[True, False])[['name','dept','salary']])

### A6. Data Types & Type Conversion
| Pattern | Syntax |
|---------|--------|
| Convert type | `df['col'].astype(int)` |
| To datetime | `pd.to_datetime(df['col'])` |
| To numeric (coerce) | `pd.to_numeric(df['col'], errors='coerce')` |
| To category | `df['col'].astype('category')` |
| Nullable int | `df['col'].astype('Int64')` (capital I) |
| Infer better types | `df.convert_dtypes()` |

> **Gotcha:** Standard int with NaN auto-converts to float64. Use `Int64` to keep ints with NaN.


In [ ]:
# Type conversion examples
sample = pd.DataFrame({'val': ['1', '2', 'bad', '4'], 'flag': [1, 0, 1, 0]})
sample['val_numeric']  = pd.to_numeric(sample['val'], errors='coerce')  # 'bad' → NaN
sample['flag_bool']    = sample['flag'].astype(bool)
sample['flag_cat']     = sample['flag'].astype('category')
print(sample)
print("\nDtypes:\n", sample.dtypes)

### A7. Missing Data Handling
| Pattern | Syntax |
|---------|--------|
| Detect NaN | `df.isna()` |
| Count per column | `df.isna().sum()` |
| Drop rows | `df.dropna(subset=['col'])` |
| Fill constant | `df.fillna(0)` |
| Fill per column | `df.fillna({'a': 0, 'b': 'unknown'})` |
| Forward fill | `df.ffill()` |
| Fill with group mean | `df.groupby('g')['c'].transform(lambda x: x.fillna(x.mean()))` |
| Interpolate | `df['col'].interpolate(method='linear')` |

> **Interview framework:** (1) Understand pattern (random vs systematic) (2) Drop if <5% (3) Impute: mean/median/mode/ffill depending on data type


In [ ]:
df_na = pd.DataFrame({
    'a': [1, np.nan, 3, np.nan, 5],
    'b': [10, 20, np.nan, 40, 50],
    'dept': ['X', 'X', 'Y', 'Y', 'X']
})
print("Original:\n", df_na)
print("\nNaN counts:", dict(df_na.isna().sum()))
print("\nAfter fillna(0):\n", df_na.fillna(0))
print("\nAfter ffill():\n", df_na.ffill())
# Fill with group mean
df_na['a_filled'] = df_na.groupby('dept')['a'].transform(lambda x: x.fillna(x.mean()))
print("\nFilled with group mean:\n", df_na)

### A8. Duplicates
| Pattern | Syntax |
|---------|--------|
| Check duplicates | `df.duplicated().sum()` |
| Show all duplicates | `df[df.duplicated(keep=False)]` |
| Drop duplicates | `df.drop_duplicates(keep='first')` |
| Subset duplicates | `df.drop_duplicates(subset=['col1','col2'])` |
| Keep row with max | `df.sort_values('val', ascending=False).drop_duplicates(subset='key', keep='first')` |

> **keep options:** `'first'` keep first, `'last'` keep last, `False` mark ALL as duplicate


In [ ]:
df_dup = pd.DataFrame({
    'id':    [1, 2, 2, 3, 3],
    'value': ['a', 'b', 'b', 'c', 'd'],
    'score': [10, 20, 20, 30, 50]
})
print("All rows:", df_dup.shape[0], " | Duplicates:", df_dup.duplicated().sum())
print("\nAfter drop_duplicates(first):\n", df_dup.drop_duplicates().reset_index(drop=True))

# Keep row with highest score per id
best = df_dup.sort_values('score', ascending=False).drop_duplicates(subset='id', keep='first')
print("\nKeep row with highest score per id:\n", best.sort_values('id'))

---
## Part B: Core Manipulation Patterns

### B1. Filtering & Selection (Advanced)
| Pattern | Syntax |
|---------|--------|
| Boolean mask | `df[df['col'] > val]` |
| Multiple conditions | `df[(cond1) & (cond2)]` |
| isin | `df[df['col'].isin([a, b])]` |
| between | `df[df['col'].between(lo, hi)]` |
| query() | `df.query('col > 5 and col2 == "x"')` |
| Top N per group | `df.groupby('g').apply(lambda x: x.nlargest(n, 'col'))` |
| np.select (multi-way) | `np.select([c1, c2], [v1, v2], default='other')` |

> `np.where` → 2 outcomes. `np.select` → multiple conditions/outcomes (if/elif/else).


In [ ]:
# Multi-condition filter
eng_high = df[(df['dept'] == 'Eng') & (df['salary'] >= 80)]
print("Eng dept, salary >= 80:\n", eng_high[['name','dept','salary']])

# np.select — 3-way conditional
conditions = [df['salary'] >= 90, df['salary'] >= 80]
choices    = ['Senior',            'Mid']
df2 = df.copy()
df2['level'] = np.select(conditions, choices, default='Junior')
print("\nnp.select level column:\n", df2[['name','salary','level']])

# top-2 per dept
top2 = df.groupby('dept').apply(lambda x: x.nlargest(2, 'salary'), include_groups=False).reset_index(level=0)
print("\nTop 2 salary per dept:\n", top2[['dept','name','salary']])

### B2. GroupBy & Aggregation
| Pattern | Syntax |
|---------|--------|
| Basic agg | `df.groupby('g')['col'].mean()` |
| Multiple agg | `.agg(['min','max','mean'])` |
| Named agg | `.agg(name=('col','func'))` |
| Transform (broadcast) | `.groupby('g')['col'].transform('sum')` |
| Filter groups | `.groupby('g').filter(lambda x: x['c'].mean() > val)` |
| Rank in group | `.groupby('g')['col'].rank(method='dense')` |

> **Key distinction — agg vs transform vs filter:**
> - `agg()` → ONE row per group (reduces)
> - `transform()` → SAME shape as input (broadcasts back)
> - `filter()` → returns subset of rows where group meets condition


In [ ]:
# Named aggregation
summary = df.groupby('dept').agg(
    headcount   = ('name',   'count'),
    avg_salary  = ('salary', 'mean'),
    max_salary  = ('salary', 'max'),
    avg_score   = ('score',  'mean')
).round(2)
print("GroupBy agg:\n", summary)

# Transform — dept average broadcast to each row
df2 = df.copy()
df2['dept_avg_salary'] = df2.groupby('dept')['salary'].transform('mean')
df2['vs_dept_avg'] = df2['salary'] - df2['dept_avg_salary']
print("\nTransform (vs dept avg):\n", df2[['name','dept','salary','dept_avg_salary','vs_dept_avg']])

# Filter groups — keep only depts where avg salary > 82
print("\nFilter — depts with avg salary > 82:")
print(df.groupby('dept').filter(lambda x: x['salary'].mean() > 82)[['name','dept','salary']])

### B3. Window Functions
| Pattern | Syntax |
|---------|--------|
| Rolling window | `.rolling(n, min_periods=1).mean()` |
| Lag (previous row) | `.groupby('g')['col'].shift(1)` |
| Lead (next row) | `.groupby('g')['col'].shift(-1)` |
| Pct change | `.groupby('g')['col'].pct_change()` |
| Expanding (cumulative) | `.expanding().mean()` |
| EWMA | `.ewm(span=N).mean()` |
| Cumcount (row number) | `.groupby('g').cumcount() + 1` |

**SQL → Pandas:**
| SQL | Pandas |
|-----|--------|
| `LAG(col,1) OVER(...)` | `.shift(1)` |
| `ROW_NUMBER() OVER(...)` | `.groupby('g').cumcount() + 1` |
| `RANK() OVER(...)` | `.rank(method='min')` |
| `DENSE_RANK() OVER(...)` | `.rank(method='dense')` |


In [ ]:
sales_data = pd.DataFrame({
    'emp_id':    [1,1,1,1,2,2,2],
    'sale_date': pd.to_datetime(['2024-01','2024-02','2024-03','2024-04',
                                 '2024-01','2024-02','2024-03']),
    'amount':    [100,120,90,150, 200,180,220]
}).sort_values(['emp_id','sale_date'])

sales_data['row_num']   = sales_data.groupby('emp_id').cumcount() + 1
sales_data['lag_1']     = sales_data.groupby('emp_id')['amount'].shift(1)
sales_data['rolling_2'] = sales_data.groupby('emp_id')['amount'].transform(
    lambda x: x.rolling(2, min_periods=1).mean())
sales_data['pct_change']= sales_data.groupby('emp_id')['amount'].pct_change().round(3)

sales_data

### B4. Pivot, Reshape & Restructure
| Pattern | Syntax |
|---------|--------|
| Pivot table | `pd.pivot_table(df, values='v', index='r', columns='c', aggfunc='sum')` |
| Melt (wide→long) | `pd.melt(df, id_vars='id', var_name='var', value_name='val')` |
| Stack (cols→rows) | `df.stack()` |
| Unstack (rows→cols) | `df.unstack(fill_value=0)` |
| Crosstab | `pd.crosstab(df['a'], df['b'])` |
| Explode lists | `df.explode('list_col')` |
| One-hot encode | `pd.get_dummies(df, columns=['cat'])` |

> `pivot()` reshapes with NO aggregation (errors on duplicates).  
> `pivot_table()` reshapes WITH aggregation (handles duplicates).


In [ ]:
wide = pd.DataFrame({
    'emp':    ['Alice','Bob','Carol'],
    'Q1_rev': [100, 120, 90],
    'Q2_rev': [110, 130, 95],
    'Q3_rev': [105, 115, 100]
})

# Melt: wide → long
long = pd.melt(wide, id_vars='emp', var_name='quarter', value_name='revenue')
print("Melt (wide→long):\n", long)

# Pivot table demo
sales_pt = pd.DataFrame({'region': ['E','E','W','W','E'], 'prod': ['A','B','A','B','B'],
                          'amt': [100,200,150,250,120]})
print("\nPivot table:\n", pd.pivot_table(sales_pt, values='amt', index='region',
      columns='prod', aggfunc='sum', fill_value=0))

# One-hot encode
print("\nOne-hot encode dept:")
print(pd.get_dummies(df[['name','dept']], columns=['dept']).head(4))

### B5. Statistical Calculations
| Pattern | Syntax |
|---------|--------|
| Describe with percentiles | `df['col'].describe(percentiles=[.1,.25,.5,.9])` |
| Correlation | `df[cols].corr()` (Pearson default) |
| Covariance | `df[cols].cov()` |
| Quantile bins (equal freq) | `pd.qcut(df['col'], q=4, labels=[...])` |
| Custom bins (equal width) | `pd.cut(df['col'], bins=[0,30,60,100])` |
| Percentile rank | `df['col'].rank(pct=True)` |
| Z-score | `(col - col.mean()) / col.std()` |
| Weighted average | `np.average(vals, weights=wts)` |

> **qcut** = equal FREQUENCY bins (same count per bin).  
> **cut** = equal WIDTH bins (same range per bin).


In [ ]:
print("Describe salary:")
print(df['salary'].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))

print("\nCorrelation matrix:")
print(df[['salary','score','age']].corr().round(3))

df2 = df.copy()
df2['salary_quartile'] = pd.qcut(df2['salary'], q=4, labels=['Q1','Q2','Q3','Q4'])
df2['age_group']       = pd.cut(df2['age'], bins=[0,30,40,100], labels=['Young','Mid','Senior'])
df2['salary_zscore']   = ((df2['salary'] - df2['salary'].mean()) / df2['salary'].std()).round(2)
df2[['name','salary','salary_quartile','age','age_group','salary_zscore']]

### B6. Merging, Joining & Combining
| Pattern | Syntax |
|---------|--------|
| Inner join | `pd.merge(a, b, on='key', how='inner')` |
| Left join | `pd.merge(a, b, on='key', how='left')` |
| Anti join | `a[~a['key'].isin(b['key'])]` |
| Semi join | `a[a['key'].isin(b['key'])]` |
| Self join | `pd.merge(df, df, on='col', suffixes=('_a','_b'))` |
| Cross join | `pd.merge(a, b, how='cross')` |
| Concat rows | `pd.concat([a, b], ignore_index=True)` |
| Combine first | `a.combine_first(b)` (fill NaN from b) |

> **Diagnostics:** `validate='one_to_one'` · `indicator=True` (adds `_merge` column)  
> **Gotcha:** Duplicate keys cause cartesian product!


In [ ]:
employees_sm = pd.DataFrame({'emp_id': [1,2,3,4], 'name': ['A','B','C','D'], 'dept': ['Eng','Sales','Eng','HR']})
sales_sm     = pd.DataFrame({'emp_id': [1,3,3],   'amount': [500,300,400]})

# Left join + aggregate
total = sales_sm.groupby('emp_id')['amount'].sum().reset_index(name='total_sales')
left  = employees_sm.merge(total, on='emp_id', how='left').fillna({'total_sales': 0})
print("Left join with totals:\n", left)

# Anti join — employees with no sales
no_sales = employees_sm[~employees_sm['emp_id'].isin(sales_sm['emp_id'])]
print("\nAnti join (no sales):\n", no_sales)

# Concat
a = pd.DataFrame({'id': [1,2], 'v': ['x','y']})
b = pd.DataFrame({'id': [2,3], 'v': ['y','z']})
print("\nConcat + dedup:\n", pd.concat([a,b]).drop_duplicates().reset_index(drop=True))

---
## Part C: Advanced Patterns

### C1. String Operations
| Pattern | Syntax |
|---------|--------|
| Lower / Upper | `.str.lower()` / `.str.upper()` |
| Strip | `.str.strip()` |
| Contains | `.str.contains('pat', na=False)` |
| Starts / Ends with | `.str.startswith('X')` |
| Split | `.str.split('_', expand=True)` |
| Replace | `.str.replace('old', 'new', regex=False)` |
| Regex extract | `.str.extract(r'(\d+)')` |
| Length | `.str.len()` |
| Slice | `.str[0:3]` |

> `.str` methods return `NaN` for `NaN` inputs. `str.contains` uses regex by default — use `regex=False` for literals.


In [ ]:
emails = pd.DataFrame({'email': ['Alice.Smith@Corp.COM', '  bob@example.com ', 'carol123@test.org', None]})
emails['lower']    = emails['email'].str.lower().str.strip()
emails['domain']   = emails['email'].str.lower().str.extract(r'@(.+)')
emails['username'] = emails['email'].str.split('@').str[0].str.lower()
emails['length']   = emails['email'].str.len()
emails['is_corp']  = emails['email'].str.contains('corp', case=False, na=False)
emails

### C2. DateTime Operations
| Pattern | Syntax |
|---------|--------|
| Parse | `pd.to_datetime(df['col'])` |
| Year / Month / Day | `.dt.year` / `.dt.month` / `.dt.day` |
| Day of week | `.dt.dayofweek` (0=Mon) |
| Quarter | `.dt.quarter` |
| Days between | `(df['d2'] - df['d1']).dt.days` |
| Add timedelta | `df['d'] + pd.Timedelta(days=7)` |
| To period | `.dt.to_period('M')` |
| Resample | `df.set_index('d').resample('M').sum()` |

> `resample()` requires a `DatetimeIndex`. Use `df.set_index('date')` first.  
> **Freq strings:** `'D'` day · `'B'` business · `'MS'` month start · `'QS'` quarter start


In [ ]:
df2 = df.copy()
df2['year']      = df2['hire_date'].dt.year
df2['month']     = df2['hire_date'].dt.month
df2['quarter']   = df2['hire_date'].dt.quarter
df2['day_name']  = df2['hire_date'].dt.day_name()
df2['tenure_days'] = (pd.Timestamp('2024-01-01') - df2['hire_date']).dt.days
df2[['name','hire_date','year','month','quarter','day_name','tenure_days']]

### C3. Apply, Map & Vectorization
| Pattern | Syntax |
|---------|--------|
| Element-wise (Series) | `df['col'].apply(func)` |
| Row-wise | `df.apply(func, axis=1)` |
| Map with dict | `df['col'].map({'a': 1, 'b': 2})` |
| Vectorized | `df['col'] * 2` |
| Vectorized conditional | `np.where(cond, true_val, false_val)` |

**Performance hierarchy (fastest → slowest):**
1. Vectorized numpy/pandas operations  ← **always prefer**
2. `.str` / `.dt` accessor methods
3. `.apply()` with simple function
4. `.itertuples()`
5. `.iterrows()` ← **avoid (very slow)**


In [ ]:
df2 = df.copy()

# Vectorized (fast)
df2['salary_k']  = df2['salary'] / 1000

# np.where (fast)
df2['level'] = np.where(df2['salary'] >= 85, 'Senior', 'Junior')

# map with dict (fast)
dept_map = {'Eng': 'Engineering', 'Sales': 'Sales', 'HR': 'Human Resources'}
df2['dept_full'] = df2['dept'].map(dept_map)

# apply (use only when necessary)
df2['initials'] = df2['name'].apply(lambda n: n[0] + '.')

df2[['name','salary','salary_k','level','dept_full','initials']]

### C4. Method Chaining & Pipe
Method chaining produces clean, readable data pipelines:

```python
result = (
    df
    .query('age > 25')
    .assign(salary_k = lambda x: x['salary'] / 1000,
            tax      = lambda x: x['salary'] * 0.3)
    .groupby('department')
    .agg(avg_salary=('salary_k', 'mean'), headcount=('name', 'count'))
    .sort_values('avg_salary', ascending=False)
    .reset_index()
)
```

Use `.pipe()` to inject custom functions into chains:
```python
def remove_outliers(df, col, n_std=3):
    return df[np.abs(df[col] - df[col].mean()) <= n_std * df[col].std()]

result = df.pipe(remove_outliers, col='salary', n_std=2)
```


In [ ]:
def flag_outliers(df, col, n_std=2):
    """Custom pipe function — marks rows beyond n_std as outliers."""
    mean, std = df[col].mean(), df[col].std()
    df = df.copy()
    df[f'{col}_outlier'] = np.abs(df[col] - mean) > n_std * std
    return df

result = (
    df
    .query('age > 25')
    .assign(salary_k=lambda x: (x['salary'] / 10).round(1))   # scale for display
    .pipe(flag_outliers, col='salary')
    .groupby('dept')
    .agg(avg_salary=('salary', 'mean'), headcount=('name', 'count'))
    .sort_values('avg_salary', ascending=False)
    .reset_index()
    .rename(columns={'dept': 'department'})
)
result

---
## Part D: Interview Decision Tree & Quick Reference

### D1. What Pandas Method Do I Use?

| Question | Answer |
|----------|--------|
| "Per group, compute X" | `groupby('g').agg(...)` |
| "Add column showing group-level stat per row" | `groupby('g')['col'].transform('stat')` |
| "Keep only groups where condition" | `groupby('g').filter(lambda x: ...)` |
| "Rank within group" | `groupby('g')['col'].rank(method='dense')` |
| "Running/moving calculation" | Sort first, then `.rolling()` |
| "Previous/next row value" | `.shift(1)` / `.shift(-1)` |
| "Wide to long" | `pd.melt()` |
| "Long to wide" | `pd.pivot_table()` or `.unstack()` |
| "Rows NOT in another table" | `df[~df['key'].isin(other['key'])]` |
| "Weighted average" | `np.average(values, weights=weights)` |
| "Bin into categories" | Equal freq: `pd.qcut()` · Equal width: `pd.cut()` |
| "Monthly/weekly aggregation" | `.set_index('date').resample('M').sum()` |
| "Remove duplicates keeping best row" | `sort_values + drop_duplicates(keep='first')` |
| "Fill NaN with group mean" | `groupby('g')['col'].transform(lambda x: x.fillna(x.mean()))` |
| "Consecutive streaks / events" | Boolean mask + cumsum trick |
| "One-hot encode" | `pd.get_dummies(df, columns=['col'])` |
| "Split string into columns" | `df['col'].str.split('_', expand=True)` |


### D2. Top 10 Classic Interview Patterns

1. **Second highest salary per dept** → `rank(method='dense') == 2`
2. **Employees earning more than dept average** → `transform('mean')` then filter
3. **Month-over-month growth** → sort + `shift` + `(current - prev) / prev`
4. **Most recent record per user** → `sort_values('date').drop_duplicates('user', keep='last')`
5. **Detect gaps in sequential data** → `.diff()` and check where diff > expected
6. **Running total by category** → sort + `groupby('cat').cumsum()`
7. **Compare each row to group median** → `transform('median')` then subtract
8. **Top N per group with ties** → `groupby + rank(method='min') <= N`
9. **Pivot report: rows=month, cols=category** → `pivot_table(aggfunc='sum')`
10. **Consecutive events (streak detection)** → `(mask != mask.shift()).cumsum()` + `cumsum`


In [ ]:
# Classic Pattern Demo — solve all 10 with the employees + sales data
np.random.seed(42)
emp = pd.DataFrame({
    'dept':   ['Eng','Eng','Eng','Sales','Sales','HR','HR','HR'],
    'name':   ['Alice','Bob','Carol','Dave','Eve','Frank','Grace','Hank'],
    'salary': [100, 85, 70, 90, 75, 80, 95, 60]
})

# 1. Second highest salary per dept (dense rank == 2)
emp['sal_rank'] = emp.groupby('dept')['salary'].rank(ascending=False, method='dense').astype(int)
print("Pattern 1 — Second highest salary per dept:")
print(emp[emp['sal_rank'] == 2][['dept','name','salary','sal_rank']], "\n")

# 2. Employees earning above dept average
emp['dept_avg'] = emp.groupby('dept')['salary'].transform('mean')
print("Pattern 2 — Above dept average:")
print(emp[emp['salary'] > emp['dept_avg']][['dept','name','salary','dept_avg']], "\n")

# 3. Streak detection
data = pd.Series([0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1])
streak_id  = (~data.astype(bool)).cumsum()
streak_len = data.groupby(streak_id).cumsum()
print("Pattern 10 — Streak detection (value=1 streaks):")
print(pd.DataFrame({'value': data, 'streak_id': streak_id, 'streak_len': streak_len}))